# Pre-compute Embeddings

Embeds all 9,472 terms in `legal_terms.json` with each of the 6 models and
persists the results as structured `.npy` arrays. This is a **one-time** step:
all Lens experiments load pre-computed vectors from disk instead of re-running
the models.

**Output** — `data/processed/embeddings/`:
```
index.json              ordered term metadata (en, zh_canonical, domain, tier)
BGE-EN-large/vectors.npy    (9472, 1024)  float32  L2-normalized
E5-large/vectors.npy        (9472, 1024)
FreeLaw-EN/vectors.npy      (9472,  768)
BGE-ZH-large/vectors.npy   (9472, 1024)
Text2vec-large-ZH/...       (9472, 1024)
Dmeta-ZH/vectors.npy        (9472,  768)
```

**Run each cell in order.** Each model cell is independent — if one fails, skip it and continue.

> ⚠️ **First run**: the 2 Sinic models not yet in HF cache will be downloaded (~1.3GB each).
> All 4 models already in cache load in seconds.

**Expected time per model** (M4, CPU): ~3–8 min for 9,472 terms

In [ ]:
import sys, os, json, time
from pathlib import Path
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from shared.embeddings import EmbeddingClient, load_precomputed

CONFIG      = ROOT / "models" / "config.yaml"
LEGAL_TERMS = ROOT / "data" / "processed" / "legal_terms.json"
EMB_DIR     = ROOT / "data" / "processed" / "embeddings"

# ── Load dataset ─────────────────────────────────────────────────────────────
data  = json.loads(LEGAL_TERMS.read_text(encoding="utf-8"))
terms = data["terms"] if isinstance(data, dict) else data
en_texts  = [t["en"]           for t in terms]
zh_texts  = [t["zh_canonical"] for t in terms]

# ── Client (device=mps for fast inference; change to cpu if MPS causes issues)
client = EmbeddingClient(CONFIG, device="mps")

# ── Stats ──────────────────────────────────────────────────────────────────
from collections import Counter
tiers   = Counter(t["tier"]   for t in terms)
domains = Counter(t["domain"] for t in terms if t["domain"])

print(f"Dataset        : {LEGAL_TERMS.name}")
print(f"Total terms    : {len(terms):,}")
print(f"Tiers          : {dict(tiers)}")
print(f"Domains (core) : {dict(domains)}")
print(f"Device         : {client._device}")
print(f"Output dir     : {EMB_DIR}")


In [ ]:
# ── Write shared index (term order is fixed from here on) ───────────────────
import hashlib
from datetime import date

EMB_DIR.mkdir(parents=True, exist_ok=True)

index = [
    {"en": t["en"], "zh_canonical": t["zh_canonical"],
     "domain": t.get("domain"), "tier": t["tier"]}
    for t in terms
]
(EMB_DIR / "index.json").write_text(
    json.dumps(index, indent=2, ensure_ascii=False), encoding="utf-8"
)

SOURCE_SHA256 = hashlib.sha256(LEGAL_TERMS.read_bytes()).hexdigest()
print(f"index.json written  ({len(index)} entries)")
print(f"source sha256: {SOURCE_SHA256[:16]}...")

In [ ]:
# ── Helper ──────────────────────────────────────────────────────────────────
def embed_and_save(model_label, texts, skip_if_exists=True):
    spec = next(s for s in client.all_specs if s.label == model_label)
    out  = EMB_DIR / model_label
    vec_path  = out / "vectors.npy"
    meta_path = out / "meta.json"

    if skip_if_exists and vec_path.exists() and meta_path.exists():
        existing = json.loads(meta_path.read_text())
        if existing.get("source_sha256") == SOURCE_SHA256:
            print(f"[{model_label}] Already computed — loading from disk.")
            return np.load(vec_path)

    out.mkdir(parents=True, exist_ok=True)
    print(f"[{model_label}] Embedding {len(texts):,} texts (lang={spec.lang}, dim={spec.dim}) ...")

    t0   = time.perf_counter()
    vecs = client.embed(texts, spec.id, use_cache=False)
    elapsed = time.perf_counter() - t0

    np.save(vec_path, vecs)
    meta_path.write_text(json.dumps({
        "model_id":      spec.id,
        "model_label":   model_label,
        "lang":          spec.lang,
        "dim":           spec.dim,
        "n_terms":       len(texts),
        "date":          date.today().isoformat(),
        "elapsed_s":     round(elapsed, 2),
        "source_sha256": SOURCE_SHA256,
    }, indent=2), encoding="utf-8")

    print(f"[{model_label}] ✓  shape={vecs.shape}  time={elapsed:.1f}s  → {vec_path}")
    return vecs

print("Helper ready.")

---
## WEIRD — Slot 1: BGE-EN-large
Already in HF cache. Should load in seconds.

In [ ]:
bge_en = embed_and_save("BGE-EN-large", en_texts)

## WEIRD — Slot 2: E5-large
Already in HF cache.

In [ ]:
e5 = embed_and_save("E5-large", en_texts)

## WEIRD — Slot 3: FreeLaw-EN
Already in HF cache.

In [ ]:
freelaw = embed_and_save("FreeLaw-EN", en_texts)

---
## Sinic — Slot 1: BGE-ZH-large
Already in HF cache.

In [ ]:
bge_zh = embed_and_save("BGE-ZH-large", zh_texts)

## Sinic — Slot 2: Text2vec-large-ZH
⚠️ **Will download ~1.3GB** on first run.

In [ ]:
text2vec = embed_and_save("Text2vec-large-ZH", zh_texts)

## Sinic — Slot 3: Dmeta-ZH
⚠️ **Will download** on first run.

In [ ]:
dmeta = embed_and_save("Dmeta-ZH", zh_texts)

---
## Verification

Check all 6 model directories exist and shapes are consistent with `index.json`.

In [ ]:
N = len(index)
print(f"{'Model':<22} {'Shape':<16} {'‖v̄‖ mean':>10}  {'Size (MB)':>10}  Status")
print("-" * 75)

all_ok = True
for spec in client.all_specs:
    vec_path = EMB_DIR / spec.label / "vectors.npy"
    if not vec_path.exists():
        print(f"{spec.label:<22} {'MISSING':<16}")
        all_ok = False
        continue
    vecs = np.load(vec_path)
    mean_norm = float(np.linalg.norm(vecs, axis=1).mean())
    size_mb   = vec_path.stat().st_size / 1e6
    shape_ok  = vecs.shape == (N, spec.dim)
    norm_ok   = abs(mean_norm - 1.0) < 1e-3
    status    = "✓" if shape_ok and norm_ok else "✗"
    if not (shape_ok and norm_ok):
        all_ok = False
    print(f"{spec.label:<22} {str(vecs.shape):<16} {mean_norm:>10.6f}  {size_mb:>10.1f}  {status}")

print()
if all_ok:
    print("✅ All 6 models verified. Pipeline ready for Lens I–V.")
else:
    print("❌ Some models missing or invalid. Re-run the failing cells.")

## Usage in experiments

Every Lens notebook loads pre-computed vectors with a single call:

```python
from shared.embeddings import load_precomputed

EMB_DIR = ROOT / "data" / "processed" / "embeddings"

vecs_en, index = load_precomputed("BGE-EN-large", EMB_DIR)
vecs_zh, _     = load_precomputed("BGE-ZH-large",  EMB_DIR)

# Filter to core terms only
core_mask = [i for i, t in enumerate(index) if t["tier"] == "core"]
core_en   = vecs_en[core_mask]   # (397, 1024)
core_zh   = vecs_zh[core_mask]   # (397, 1024)
```